In [3]:
import numpy as np
import strawberryfields as sf
import matplotlib.pyplot as plt
from scipy.integrate import simps
# ── Parameter global Bose-Hubbard ────────────────────────────────────────────

cutoff = 10  # Truncation dimensi Fock space per mode.
             # Artinya setiap qumode hanya bisa menampung 0 sampai 9 foton.
             # Analogi: memotong deret Taylor di suku ke-10.
             # Makin besar cutoff → makin akurat tapi makin lambat (O(cutoff²) memori).

J = 1        # Hopping integral: seberapa mudah foton melompat antar node.
             # J besar → foton mobile (superfluid), J kecil → foton terlokalisir.

U = 1.5      # On-site interaction: energi penalti jika dua foton di node yang sama.
             # U >> J → Mott insulator (foton tidak mau berbagi node).
             # U << J → superfluid (foton bebas bergerak).

k = 20       # Jumlah Trotter steps. Makin banyak → error O(t²/k) makin kecil.
             # Trade-off: lebih akurat tapi circuit lebih dalam (lebih lambat).

t = 1.086    # Total waktu propagasi dalam satuan atomik (a.u.).
             # Nilai ini dipilih untuk menunjukkan dinamika yang menarik.

# Parameter gate dari dekomposisi Trotter Bose-Hubbard (Eq. 46 paper):
theta = -J * t / k   # Sudut BSgate: negatif karena hopping term memiliki tanda negatif
                      # dalam eksponensial e^{-iHt}.
r = -U * t / (2 * k) # Parameter Kgate: faktor 2k karena split ke dua node,
                      # negatif karena on-site term U/2·(n²-n).

print(f"Parameter BSgate: θ = {theta:.4f}")
print(f"Parameter Kgate:  r = {r:.4f}")

# ── State awal ───────────────────────────────────────────────────────────────

# State awal: |2,0⟩ — dua foton di node 1 (mode q[0]), nol foton di node 2.
# Array berukuran [cutoff, cutoff] = [10, 10] merepresentasikan
# koefisien c_{i,j} dalam ekspansi |ψ⟩ = Σ c_{i,j} |i,j⟩.
ket = np.zeros([cutoff, cutoff], dtype=np.complex64)
ket[2, 0] = 1.0 + 0.0j  # Hanya satu elemen non-nol: 2 foton di mode 0, 0 foton di mode 1.
                          # Ini menyatakan state Fock |2,0⟩ secara eksplisit.

print(f"\nState awal |ψ₀⟩: ket[2,0] = {ket[2,0]}")
print(f"Verifikasi normalisasi: Σ|c|² = {np.sum(np.abs(ket)**2):.4f}")  # harus = 1.0

# ── Definisi circuit Strawberry Fields ────────────────────────────────────────

prog = sf.Program(2)  # Deklarasi program dengan 2 qumode (2 mode foton).
                       # Analogi dengan QuantumCircuit(2) di Qiskit, tapi untuk CV.

with prog.context as q:  # Context manager: semua operasi di dalam blok ini
                          # direkam ke dalam program (tidak langsung dieksekusi).

    sf.ops.Ket(ket) | q  # Inisialisasi seluruh sistem (q[0] dan q[1]) dengan
                          # statevector ket yang sudah didefinisikan di atas.
                          # Ini operasi "state preparation" — analog qc.prepare_state() di Qiskit.
                          # | adalah operator "pipe" khas Strawberry Fields (bukan bitwise OR!).

    for i in range(k):   # Loop sebanyak k=20 Trotter steps.
                          # Tiap iterasi = satu layer Trotterisasi.

        # Step 1: BSgate mengimplementasikan hopping term J·(â†₁â₂ + â†₂â₁).
        # BSgate(θ, ϕ): beamsplitter dengan transmittivity angle θ dan phase ϕ=π/2.
        # Efek fisik: interferensi antara dua mode foton — seperti cermin setengah pantul.
        # θ = -Jt/k mengodekan seberapa banyak "hopping" terjadi per Trotter step.
        sf.ops.BSgate(theta, np.pi / 2) | (q[0], q[1])

        # Step 2: Kgate mengimplementasikan on-site interaction U/2·n̂².
        # K(κ) = exp(iκn̂²) — gate non-linear (non-Gaussian).
        # Ini yang mengharuskan kita pakai backend Fock, bukan Gaussian.
        # Analogi: seperti efek Kerr dalam optik non-linear — intensitas cahaya mempengaruhi fasanya sendiri.
        sf.ops.Kgate(r) | q[0]   # on-site interaction untuk mode 0
        sf.ops.Kgate(r) | q[1]   # on-site interaction untuk mode 1

        # Step 3: Rgate mengoreksi fase dari term -U/2·n̂ dalam Hamiltonian.
        # R(-r) = exp(-ir·n̂) — rotation gate, mengimplementasikan e^{iUt/(2k)·n̂}.
        # Ini diperlukan karena Bose-Hubbard memiliki term linear n̂ yang ikut
        # dalam on-site interaction: U/2·(n̂²-n̂).
        sf.ops.Rgate(-r) | q[0]  # phase correction mode 0
        sf.ops.Rgate(-r) | q[1]  # phase correction mode 1

# ── Eksekusi ─────────────────────────────────────────────────────────────────

# Pilihan backend: 'fock' karena ada Kerr gate (non-Gaussian).
# Backend lain yang tersedia:
#   'gaussian' — untuk Gaussian states saja (lebih cepat, tidak bisa handle Kerr)
#   'bosonic'  — untuk Wigner negativity
#   'tf'       — TensorFlow backend untuk optimasi (dipakai di 5.2)
eng = sf.Engine('fock', backend_options={"cutoff_dim": cutoff})

# Jalankan program dan ambil state hasil.
# eng.run() mengembalikan objek Result; .state adalah quantum state setelah eksekusi.
result = eng.run(prog)
state  = result.state

# ── Probabilitas Fock ────────────────────────────────────────────────────────

# fock_prob([n0, n1]): probabilitas mengukur n0 foton di mode 0 dan n1 foton di mode 1.
# Hasilnya adalah |⟨n0,n1|ψ⟩|².
P_02 = state.fock_prob([0, 2])  # P(|0,2⟩): semua foton pindah ke node 2
P_11 = state.fock_prob([1, 1])  # P(|1,1⟩): foton terdistribusi merata
P_20 = state.fock_prob([2, 0])  # P(|2,0⟩): foton tetap di node 1 (state awal)

print(f"\nProbabilitas Fock setelah propagasi:")
print(f"  P(|0,2⟩) = {P_02:.6f}")
print(f"  P(|1,1⟩) = {P_11:.6f}")
print(f"  P(|2,0⟩) = {P_20:.6f}")
print(f"  Total    = {P_02 + P_11 + P_20:.6f}  (harus ≈ 1.0 → konservasi partikel)")

Parameter BSgate: θ = -0.0543
Parameter Kgate:  r = -0.0407

State awal |ψ₀⟩: ket[2,0] = (1+0j)
Verifikasi normalisasi: Σ|c|² = 1.0000

Probabilitas Fock setelah propagasi:
  P(|0,2⟩) = 0.522401
  P(|1,1⟩) = 0.235653
  P(|2,0⟩) = 0.241946
  Total    = 1.000000  (harus ≈ 1.0 → konservasi partikel)


In [4]:
import math  # Gunakan math.factorial, BUKAN np.math.factorial yang deprecated

def hopping_term(statevec, J, cutoff):
    """
    Menghitung expectation value dari hopping term J: ⟨ψ|Ĵ|ψ⟩
    dimana Ĵ = J·(â†₁â₂ + â†₂â₁).

    Derivasi:
    ⟨â†₁â₂⟩ = Σᵢⱼ c*ᵢⱼ · √(i+1)·√j · cᵢ₊₁,ⱼ₋₁
    karena â†|i⟩ = √(i+1)|i+1⟩ dan â|j⟩ = √j|j-1⟩.

    Analogi: seperti menghitung korelasi antar tetangga dalam rantai spin.
    """
    svc = np.conj(statevec)  # Konjugat untuk bra ⟨ψ| — diperlukan untuk inner product.
    hop = 0.0                 # Akumulator nilai expectation.

    for i in range(cutoff):
        if i == 0:
            # Untuk i=0: hanya kontribusi dari â†₂â₁ (foton dari mode 1 ke mode 0).
            # â†₁|0⟩ = |1⟩, â₂|j⟩ = √j|j-1⟩.
            for j in range(1, cutoff):
                # c*(0,j) · √((0+1)·j) · c(1,j-1)
                hop += svc[i, j] * np.sqrt((i + 1) * j) * statevec[i + 1, j - 1]
        else:
            # Untuk i>0: kontribusi dari â†₁â₂ (foton dari mode 0 ke mode 1).
            # â†₂|i⟩|j⟩: mode 1 dapatkan foton dari mode 0.
            for j in range(cutoff - 1):
                # c*(i,j) · √(i·(j+1)) · c(i-1,j+1)
                hop += svc[i, j] * np.sqrt(i * (j + 1)) * statevec[i - 1, j + 1]

            # Juga kontribusi dari â†₁â₂ arah sebaliknya jika i < cutoff-1.
            if i < cutoff - 1:
                for x in range(1, cutoff):
                    hop += svc[i, x] * np.sqrt((i + 1) * x) * statevec[i + 1, x - 1]

    return np.abs(hop * J)   # Nilai absolut karena energi harus real positif.


def on_site(statevec, U, cutoff):
    """
    Menghitung expectation value dari on-site term: ⟨ψ|Û/2|ψ⟩
    dimana Û = U·(n̂₁²-n̂₁+n̂₂²-n̂₂).

    Derivasi:
    ⟨n̂₁⟩ = Σᵢⱼ |cᵢⱼ|² · i  (karena n̂|i⟩ = i|i⟩)
    ⟨n̂₁²⟩ = Σᵢⱼ |cᵢⱼ|² · i²

    BUG FIX dari paper: paper menggunakan np.abs(statevec[i,j]) yang menghasilkan |c|,
    bukan np.abs(statevec[i,j])**2 yang menghasilkan |c|² = probabilitas yang benar.
    Untuk expectation value kuantum, kita selalu butuh ρᵢⱼ = |cᵢⱼ|².
    """
    on_site_val = 0.0

    for i in range(cutoff):
        for j in range(cutoff):
            # BUG FIX: ** 2 di sini WAJIB — ini probabilitas |c_{ij}|²
            prob = np.abs(statevec[i, j]) ** 2

            # Term energi: i²+j²-i-j berasal dari n̂₁²-n̂₁+n̂₂²-n̂₂
            # dengan eigenvalue masing-masing i² - i = i(i-1) dan j² - j = j(j-1).
            on_site_val += prob * (i**2 + j**2 - i - j)

    return np.abs(on_site_val * U / 2)  # Faktor U/2 dari Hamiltonian.


# ── Hitung energi ─────────────────────────────────────────────────────────────

# Ambil statevector dari state hasil simulasi.
# state.ket() mengembalikan array [cutoff, cutoff] = tensor koefisien c_{i,j}.
sket = state.ket()
print(f"Shape statevector: {sket.shape}")  # harus (10, 10)

# Normalisasi: inner product ⟨ψ|ψ⟩ = Σ|c_{ij}|² — seharusnya ≈ 1.
inner_prod = np.abs(np.vdot(sket.flatten(), sket.flatten()))
print(f"Normalisasi |⟨ψ|ψ⟩|: {inner_prod:.6f}")  # Harus ≈ 1.0

# Total energi: E = (⟨Ĵ⟩ + ⟨Û/2⟩) / ⟨ψ|ψ⟩
# Pembagian dengan inner_prod untuk normalisasi (jaga-jaga drift numerik).
hop_E  = hopping_term(sket, J, cutoff)
site_E = on_site(sket, U, cutoff)
energy_total = (hop_E + site_E) / inner_prod

print(f"\nKomponen energi:")
print(f"  Hopping term ⟨Ĵ⟩   = {hop_E:.6f}")
print(f"  On-site term ⟨Û/2⟩ = {site_E:.6f}")
print(f"  Total energi E     = {energy_total:.6f}")
print(f"  Paper melaporkan   = 2.135643 (dengan bug on_site)")
print(f"  Nilai terkoreksi seharusnya berbeda karena bug on_site diperbaiki")

Shape statevector: (10, 10)
Normalisasi |⟨ψ|ψ⟩|: 1.000000

Komponen energi:
  Hopping term ⟨Ĵ⟩   = 0.313663
  On-site term ⟨Û/2⟩ = 1.146521
  Total energi E     = 1.460184
  Paper melaporkan   = 2.135643 (dengan bug on_site)
  Nilai terkoreksi seharusnya berbeda karena bug on_site diperbaiki
